# Session 3: Teaching the Engine to Learn

In Session 2, we built an adaptive rebalancing engine that reads market sentiment and adjusts allocation in real time. The engine works, but it has two frozen components: the SIM parameters $(\alpha_i, \beta_i, \sigma_{\varepsilon,i})$ were calibrated once from historical data, and the CES elasticity $\eta(\lambda_t)$ follows a hand-designed heuristic. 

__Challenge:__ Both assumptions break when the market enters a new regime. In this session, we replace frozen parameters with online estimation and replace the heuristic with a learning algorithm. We then validate the results against formal deployment gates. Two classic AI techniques enter the engine in this session. 
* **Exponentially weighted least squares** is the streaming counterpart of batch OLS, mathematically equivalent to stochastic gradient descent with an exponentially decaying step size; it is the building block of every online-learning system from Kalman filters to adaptive signal processing. 
* **Multi-armed bandits** are the canonical reinforcement-learning problem: an agent repeatedly selects from a discrete action set, observes a stochastic reward, and trades off exploration against exploitation. The epsilon-greedy policy with decaying exploration achieves provably sublinear regret: the learner converges on the best arm while never stopping exploring entirely. 

Both techniques are mature AI; their application to the preference-weight and elasticity-selection problems here is the session's contribution.

> __Learning Objectives:__
>
> By the end of this session, you will be able to:
> * __Online SIM Parameter Estimation:__ Implement exponentially weighted least squares (EWLS) to update the SIM parameters daily as new market data arrives. Control the speed-stability trade-off through the half-life decay factor and prior weight.
> * __Bandit Elasticity Learning Per Regime:__ Apply an epsilon-greedy multi-armed bandit to learn the best CES elasticity from a discrete grid, with a separate bandit instance per sentiment regime. Compare the learned elasticity values to the Session 2 heuristic formula.
> * __Validation Report and Compliance Configuration:__ Evaluate strategy performance against five pass/fail deployment gates that cover risk-adjusted growth, drawdown, failure rate, wealth versus the frozen baseline, and median net present value. Export the resulting compliance configuration that defines the pre-approved operating limits for Session 4's production system.

Let's teach the engine to learn.

___

## Examples

The example notebooks below accompany this lecture. They split into __core__ examples that we should run live alongside the lecture, and __optional__ examples that extend the core material and can be studied afterward at our own pace.

### Core (tonight)

These three notebooks carry the session's online-learning chain: replace the engine's frozen SIM calibration with daily EWLS updates, swap the hand-designed CES elasticity heuristic for a bandit-learned map, and apply five formal validation gates that gate-keep whether the upgraded engine can move to Session 4's production system.

> __Run live with the lecture:__
>
> * [▶ Let's replay the Cobb-Douglas engine with online EWLS parameter updates](eCornell-AI-Finance-S3-Example-Core-EWLSEngineReplay-May-2026.ipynb). We replay the Session 2 engine on a drifted Monte Carlo path ensemble with EWLS re-estimating $(\alpha_i, \beta_i, \sigma_{\varepsilon,i})$ daily, compare the frozen and online performance distributions, and zoom into single paths to see how parameter estimates adapt to regime shifts.
> * [▶ Let's learn the CES elasticity per regime with an epsilon-greedy bandit](eCornell-AI-Finance-S3-Example-Core-BanditEtaLearning-May-2026.ipynb). We run the bandit over a discrete $\eta$ grid with separate instances per sentiment-regime bin, visualize convergence and regret, then backtest the learned $\eta$ map against the Session 2 heuristic across the path ensemble.
> * [▶ Let's evaluate the engine against five deployment gates](eCornell-AI-Finance-S3-Example-Core-ValidationReport-May-2026.ipynb). We score the EWLS engine, bandit-learned $\eta$ engine, frozen baseline, and buy-and-hold against Sharpe, drawdown, failure-rate, wealth-vs-frozen, and median-NPV thresholds, produce a pass/fail validation report, and export the compliance configuration that Session 4's production system enforces.
>

### Optional (self-study)

This notebook extends the bandit framework to a harder problem than elasticity selection: combinatorial asset selection over $2^K - 1$ non-empty subsets of the ticker universe. It is not required to reach the Session 4 hand-off.

> __Extension material (self-study):__
>
> * [▶ Let's try combinatorial asset selection with a $2^K - 1$ arm bandit](eCornell-AI-Finance-S3-Example-Optional-TickerPickerBandit-May-2026.ipynb). We train an epsilon-greedy bandit over subsets of the ticker universe, backtest the selected portfolio across the Monte Carlo ensemble, and evaluate whether the bandit-selected engine passes the same deployment gates from the core validation example.
>
> Study this when we want to see how bandit exploration scales as the action space grows exponentially.

___

## Notation

This lecture extends the SIM and CES machinery from Sessions 1 and 2. The symbols below are inherited with their definitions from the linked sections; we restate only the role each plays here.

> __Inherited from prior sessions:__
>
> * $g_i, g_{\mathrm{mkt}}, \varepsilon_i$: asset and market continuously compounded growth rates (CCGR) and SIM residuals; defined in [S1 § From Prices to Growth Rates](../session-1/eCornell-AI-Finance-S1-Lecture-StressTestingMinVariancePortfolios-May-2026.ipynb#from-prices-to-growth-rates) and [S1 § Modern Portfolio Theory](../session-1/eCornell-AI-Finance-S1-Lecture-StressTestingMinVariancePortfolios-May-2026.ipynb#modern-portfolio-theory-mpt).
> * $\alpha_i, \beta_i, \sigma_{\varepsilon,i}$: SIM intercept, slope, and residual standard deviation for asset $i$; calibrated batch-style in S1 and S2, re-estimated online in this session. Defined in [S2 § Cobb-Douglas Utility](../session-2/eCornell-AI-Finance-S2-Lecture-AI-RebalancingEngine-May-2026.ipynb#cobb-douglas-utility).
> * $\mathbf{w}, w_i, \gamma_i$: portfolio weight vector, asset $i$ weight, and Cobb-Douglas preference weight with $\gamma_i \in (-1, 1)$. See [S1 § Modern Portfolio Theory](../session-1/eCornell-AI-Finance-S1-Lecture-StressTestingMinVariancePortfolios-May-2026.ipynb#modern-portfolio-theory-mpt) and [S2 § Cobb-Douglas Utility](../session-2/eCornell-AI-Finance-S2-Lecture-AI-RebalancingEngine-May-2026.ipynb#cobb-douglas-utility).
> * $\lambda_t$: market sentiment signal from the EMA crossover; bearish when $\lambda_t > 0$, bullish when $\lambda_t < 0$. Defined in [S2 § Cobb-Douglas Utility](../session-2/eCornell-AI-Finance-S2-Lecture-AI-RebalancingEngine-May-2026.ipynb#cobb-douglas-utility).
> * $\eta, \eta_{\min}, \eta_{\max}$: CES elasticity of substitution and its bounds (defaults $\eta_{\min} = 0.5, \eta_{\max} = 5.0$). Defined in [S2 § CES](../session-2/eCornell-AI-Finance-S2-Lecture-AI-RebalancingEngine-May-2026.ipynb#ces-constant-elasticity-of-substitution).

__New in this session:__ $\hat{\alpha}, \hat{\beta}, \hat{\sigma}_{\varepsilon}$ are the EWLS running estimates of the SIM parameters, written with hats to distinguish them from the unhatted historical-calibration values $(\alpha_0, \beta_0, \sigma_{\varepsilon,0})$ used to seed the recursion.

___


## The Problem: Frozen Parameters and Heuristic Elasticity

In Session 2, the engine computes preference weights $\gamma_i$ from SIM parameters $(\alpha_i, \beta_i, \sigma_{\varepsilon,i})$ estimated once from a historical training window. These parameters are frozen for the life of the backtest. The adaptive CES elasticity $\eta(\lambda_t)$ follows a formula chosen by us (based on some intuition), but not learned from data.

> __Two gaps in the Session 2 engine:__
>
> 1. **Parameter drift.** The SIM triple $(\alpha_i, \beta_i, \sigma_{\varepsilon,i})$ was calibrated from a training window and never updated. If the market enters a new regime, the frozen parameters send stale signals to the preference weight formula. Alpha estimates from a bull market are misleading in a bear market; beta estimates from low-volatility periods understate risk during a crisis.
> 2. **Heuristic elasticity.** The adaptive elasticity $\eta(\lambda_t) = \eta_{\min} + (\eta_{\max} - \eta_{\min})/(1 + |\lambda_t|)$ was designed by hand. It applies the same monotonic mapping regardless of regime, and there is no guarantee this functional form is optimal. A bearish regime might call for a different eta than a bullish regime at the same $|\lambda|$.

Session 3 fixes the first gap with exponentially weighted least squares (online parameter estimation) and the second with a multi-armed bandit (learned elasticity per regime).

## EWLS: Online SIM Parameter Estimation

The Single Index Model regression $g_i = \alpha_i + \beta_i \, g_{\mathrm{mkt}} + \varepsilon_i$ can be re-estimated online as new data arrives. Exponentially weighted least squares (EWLS) discounts old observations with a decay factor so the estimates track regime shifts without a sliding window or batch re-calibration. The path from "rerun batch WLS every day" to "constant-time update plus a $2 \times 2$ solve" is short, and it falls out of the structure of the weighted normal equations.

### Deriving the EWLS Recursion

We want to track the SIM parameters $(\alpha_i, \beta_i)$ for asset $i$ as new market days arrive, without storing every past observation. The natural starting point is the regression itself, written in vector form. Fix asset $i$ and at each integer step $s = 1, 2, \ldots, t$ observe the asset CCGR $g_{i,s}$ and the market CCGR $g_{\mathrm{mkt},s}$. Stack the regressors with an intercept into the augmented vector $\mathbf{x}_s = [1,\; g_{\mathrm{mkt},s}]^{\top} \in \mathbb{R}^{2}$, write the SIM parameters as $\boldsymbol{\theta}_i = [\alpha_i,\; \beta_i]^{\top}$, and write the target as the scalar $y_s = g_{i,s}$. The single-asset SIM regression collapses to:

$$y_s = \mathbf{x}_s^{\top}\boldsymbol{\theta}_i + \varepsilon_{i,s}$$

Old observations should fade as new ones arrive, so pick a half-life $h$ in trading days and define the decay factor $\delta = 2^{-1/h} \in (0,1)$ (we use $\delta$ rather than $\rho$ here so the symbol does not collide with the regime label $\rho_t$ in the bandit section, and we reserve $\eta$ for the CES elasticity). The weight at time $t$ on the observation made at step $s \leq t$ is then:

$$w_{s,t} = \delta^{\,t-s}$$

An observation made $h$ steps before $t$ carries half the weight of today's observation since $\delta^{h} = 1/2$. The exponentially weighted least squares estimate of $\boldsymbol{\theta}_i$ at time $t$ minimizes the corresponding weighted squared residual:

$$L_t(\boldsymbol{\theta}) = \sum_{s=1}^{t} w_{s,t}\,(y_s - \mathbf{x}_s^{\top}\boldsymbol{\theta})^{2}$$

Setting $\nabla_{\boldsymbol{\theta}} L_t = \mathbf{0}$ produces the weighted normal equations $\mathbf{A}_t\,\boldsymbol{\theta} = \mathbf{b}_t$, where the two weighted moments needed for the optimum are $\mathbf{A}_t \in \mathbb{R}^{2 \times 2}$ and $\mathbf{b}_t \in \mathbb{R}^{2}$, and we keep one extra weighted scalar moment $c_t \in \mathbb{R}$ that will return when we estimate the residual variance:

$$\mathbf{A}_t = \sum_{s=1}^{t} w_{s,t}\,\mathbf{x}_s\mathbf{x}_s^{\top}, \qquad \mathbf{b}_t = \sum_{s=1}^{t} w_{s,t}\,\mathbf{x}_s y_s, \qquad c_t = \sum_{s=1}^{t} w_{s,t}\,y_s^{2}$$

Everything WLS needs is encoded in these three running quantities. The $(1,1)$ entry of $\mathbf{A}_t$ is the effective sample weight $S_{w,t} = \sum_s w_{s,t}$ since $\mathbf{x}_s\mathbf{x}_s^{\top}$ has a $1$ in the top-left; the off-diagonal carries $\sum_s w_{s,t}\,g_{\mathrm{mkt},s}$, and the $(2,2)$ entry carries $\sum_s w_{s,t}\,g_{\mathrm{mkt},s}^{2}$. Each moment is a weighted sum, and when time advances from $t-1$ to $t$ every old weight $w_{s,t-1}$ becomes $\delta\cdot w_{s,t-1}$ in $w_{s,t}$ while the new observation $(\mathbf{x}_t, y_t)$ enters with weight $1$, so the recursion is mechanical:

$$\boxed{\mathbf{A}_t \leftarrow \delta\,\mathbf{A}_{t-1} + \mathbf{x}_t\mathbf{x}_t^{\top}, \qquad \mathbf{b}_t \leftarrow \delta\,\mathbf{b}_{t-1} + \mathbf{x}_t y_t, \qquad c_t \leftarrow \delta\,c_{t-1} + y_t^{2}\quad\blacksquare}$$

A $2 \times 2$ rank-one matrix add, a 2-vector add, and a scalar add: the entire past is compressed into $\mathbf{A}_t$, $\mathbf{b}_t$, $c_t$ with no raw history retained. Solving the weighted normal equations against the freshly aged $\mathbf{A}_t$ and $\mathbf{b}_t$ gives the running SIM estimate:

$$\boxed{\hat{\boldsymbol{\theta}}_{i,t} = \begin{bmatrix} \hat{\alpha}_{i,t} \\ \hat{\beta}_{i,t} \end{bmatrix} = \mathbf{A}_t^{-1}\,\mathbf{b}_t\quad\blacksquare}$$

Since $\mathbf{A}_t$ is $2 \times 2$, this is one cheap solve per day, not a full matrix inversion. The residual standard deviation falls out of the same machinery. Expanding the weighted residual sum of squares at the optimum and substituting the normal equations $\mathbf{A}_t\hat{\boldsymbol{\theta}}_{i,t} = \mathbf{b}_t$ into the quadratic term, the cross term cancels:

$$\sum_{s=1}^{t} w_{s,t}\,(y_s - \mathbf{x}_s^{\top}\hat{\boldsymbol{\theta}}_{i,t})^{2} = c_t - 2\,\hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{b}_t + \hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{A}_t\,\hat{\boldsymbol{\theta}}_{i,t} = c_t - \hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{b}_t$$

Dividing by the effective sample weight $[\mathbf{A}_t]_{1,1}$ gives the residual standard deviation estimate:

$$\boxed{\hat{\sigma}_{\varepsilon,i,t} = \sqrt{\frac{c_t - \hat{\boldsymbol{\theta}}_{i,t}^{\top}\,\mathbf{b}_t}{[\mathbf{A}_t]_{1,1}}}\quad\blacksquare}$$

Every trading day, three running moments age by $\delta$, today's $(\mathbf{x}_t, y_t)$ folds in, one $2 \times 2$ solve produces $(\hat{\alpha}_{i,t}, \hat{\beta}_{i,t})$, and a single inner product produces $\hat{\sigma}_{\varepsilon,i,t}$: constant memory, constant compute per step, no raw history retained.

### Prior Seeding and Half-Life

Two knobs control how aggressively EWLS adapts: the prior weight $W_0$ that anchors the recursion before any new data arrives, and the half-life $h$ that sets how fast old data fades.

* __Prior weight $W_0$:__ Before any observation is processed, the sufficient statistics are seeded as if $W_0$ pseudo-observations consistent with the calibrated parameters $\boldsymbol{\theta}_{i,0} = [\alpha_{i,0},\; \beta_{i,0}]^{\top}$ and residual scale $\sigma_{\varepsilon,i,0}$ had been seen, giving $\mathbf{A}_0 \approx W_0 \cdot \mathbb{E}[\mathbf{x}\mathbf{x}^{\top}]$, $\mathbf{b}_0 \approx W_0 \cdot \mathbb{E}[\mathbf{x}\mathbf{x}^{\top}]\,\boldsymbol{\theta}_{i,0}$, and $c_0$ accordingly. Large $W_0$ keeps the calibrated values dominant longer; small $W_0$ lets the first few real observations move the estimates appreciably.
* __Half-life $h$ (speed):__ A short half-life such as $h = 21$ days (one trading month) puts most of the weight on the last few weeks of data, so the estimates track regime shifts quickly but inherit single-day noise. The $h = 21$ setting is the right choice when responsiveness matters more than smoothness.
* __Half-life $h$ (stability):__ A long half-life such as $h = 126$ days (six months) averages over a much wider window and produces smooth estimates that lag genuine regime transitions. The default $h = 63$ days (one trading quarter) sits between these extremes and is the value the engine ships with for daily rebalancing.

Together $W_0$ and $h$ move the estimator along the speed-stability frontier; everything else in the recursion is mechanical.

> __Example__
>
> [▶ Let's replay the engine with EWLS parameter updates](eCornell-AI-Finance-S3-Example-Core-EWLSEngineReplay-May-2026.ipynb). We run the engine on the Session 2 Monte Carlo ensemble with frozen vs. online parameters, compare distributional metrics, and zoom into single paths to see what one realized future looks like.

___

## Reinforcement Learning Problem
Suppose we have an agent that can be in a state $s \in \mathcal{S}$ and can take an action $a \in \mathcal{A}$. After taking action $a$ in state $s$, the agent receives a reward $r$. But how does the agent learn to choose the best possible action in each state to maximize its cumulative reward over time?

<div>
    <center>
        <img src="figs/Fig-Schematic-RL.svg" width="580"/>
    </center>
</div>

In reinforcement learning, an agent interacts with an environment by observing its current state $s \in \mathcal{S}$, selecting an action $a \in \mathcal{A}$, and receiving a reward that influences its future decisions. The canonical text covering all three approaches below is [Sutton and Barto, *Reinforcement Learning: An Introduction* (2nd ed., 2018)](http://incompleteideas.net/book/the-book-2nd.html). There are many different strategies for this problem:

* __Multiplicative weights__ adjusts the probability of selecting an action based on past performance, in a principled way that guarantees the algorithm performs nearly as well as the best fixed action in hindsight, even in changing environments; that is, it minimizes regret. See [Arora, Hazan, and Kale (2012), *The Multiplicative Weights Update Method: A Meta-Algorithm and Applications*](https://theoryofcomputing.org/articles/v008a006/) for the unified survey.
* __Bandit algorithms__ operate in stateless environments. On each round, they explore different actions to estimate their rewards and adapt their action-selection strategy based on the outcomes. The modern reference is [Slivkins, *Introduction to Multi-Armed Bandits* (2019)](https://arxiv.org/abs/1904.07272); the encyclopedic treatment is [Lattimore and Szepesvári, *Bandit Algorithms* (2020)](https://tor-lattimore.com/downloads/book/book.pdf).
* __Q-learning__ is a value-based method that estimates the long-term value (utility, satisfaction, happiness, etc) of each state-action pair, enabling the agent to learn optimal behavior in environments with temporal and sequential dynamics. The original convergence proof is [Watkins and Dayan, *Q-learning*, *Machine Learning* 8 (1992)](https://link.springer.com/article/10.1007/BF00992698); the deep-network extension that brought the method into mainstream practice is [Mnih et al., *Human-level control through deep reinforcement learning*, *Nature* 518 (2015)](https://www.nature.com/articles/nature14236).

These approaches highlight different strategies for learning from interaction, but they all must balance a fundamental challenge in reinforcement learning: the tradeoff between exploring new actions to gather information and exploiting known actions to maximize reward.

___

## Multi-Armed Bandits for Elasticity Learning

The adaptive elasticity heuristic from Session 2 maps sentiment to elasticity via a formula designed by inspection. A bandit treats this as a learning problem: choose elasticity from a discrete grid at each rebalance, observe the resulting engine performance, and converge to the best choice per regime.

> __The Multi-Armed Bandit Framework:__
>
> A bandit faces a finite arm set $\mathcal{A} = \{1, 2, \ldots, K\}$ of cardinality $K$. At each round $t = 1, \ldots, T$ the learner selects an arm $a_t \in \mathcal{A}$ and observes a stochastic reward $r_t$ drawn from a fixed but unknown distribution that depends on $a_t$. Let $\mu_k = \mathbb{E}[r_t \mid a_t = k]$ denote the mean reward of arm $k$, $\mu^{\star} = \max_{k \in \mathcal{A}} \mu_k$ the mean of the optimal arm, and $\mu_{a_t}$ the mean of the arm pulled at round $t$. The goal is to minimize cumulative **regret**, the gap between the reward of the best arm in hindsight and the reward actually collected:
>
> $$\boxed{R_T = \sum_{t=1}^{T} \left(\mu^{\star} - \mu_{a_t}\right)\quad\blacksquare}$$
>
> Low regret means the learner quickly identified the best arm and exploited it. The epsilon-greedy policy balances exploration and exploitation with a decaying exploration rate.
>
> __Epsilon-Greedy with Decaying Exploration:__
>
> At round $t$, the agent explores (pulls a random arm) with probability $\varepsilon_t$ and exploits (pulls the arm with the highest estimated mean) with probability $1 - \varepsilon_t$. The exploration rate decays as:
>
> $$\boxed{\varepsilon_t = t^{-1/3} \cdot \left(K \cdot \ln t\right)^{1/3}\quad\blacksquare}$$
>
> This decay guarantees sublinear regret $R_T = O(T^{2/3} (K \ln T)^{1/3})$ (original finite-time analysis: Auer, Cesa-Bianchi, and Fischer [2002]; modern treatment: [Slivkins, *Introduction to Multi-Armed Bandits*](https://arxiv.org/abs/1904.07272)). The learner converges, but never stops exploring entirely.

This is an *online* algorithm: reward draws arrive one at a time, and the arm-mean estimates update after every pull rather than at the end of a batch. The mechanics are compact enough to state as pseudocode directly.

### Algorithm: Epsilon-Greedy Bandit (Online)

__Initialize__: Given an arm set $\mathcal{A} = \{1, 2, \ldots, K\}$, a horizon $T$, and a reward oracle that returns a stochastic reward $r_t$ when an arm is pulled, set the mean-reward estimates $\hat{\mu}_a \gets 0$ and pull counts $n_a \gets 0$ for each $a \in \mathcal{A}$.

For $t = 1, \ldots, T$ __do__:

1. Compute the exploration rate: $\varepsilon_t \gets t^{-1/3} \cdot (K \ln t)^{1/3}$.
2. Draw $p \sim \mathrm{Uniform}[0,1]$ and select an arm:
    - If $p \leq \varepsilon_t$, *explore*: sample $a_t$ uniformly from $\mathcal{A}$.
    - Otherwise, *exploit*: set $a_t \gets \arg\max_{a \in \mathcal{A}} \hat{\mu}_a$ (break ties arbitrarily).
3. Pull arm $a_t$ and observe the stochastic reward $r_t$ from the environment.
4. Update the pull count and running mean for arm $a_t$:
    $$n_{a_t} \gets n_{a_t} + 1, \qquad \hat{\mu}_{a_t} \gets \hat{\mu}_{a_t} + \frac{1}{n_{a_t}}\left(r_t - \hat{\mu}_{a_t}\right)$$

__Output__: Return the mean-reward estimates $\{\hat{\mu}_a\}_{a \in \mathcal{A}}$, pull counts $\{n_a\}_{a \in \mathcal{A}}$, and the action-reward history $\{(a_t, r_t)\}_{t=1}^{T}$.

The sample-average update in step 4 gives every observation of arm $a$ equal weight; replacing $1/n_{a_t}$ with a constant learning rate $\alpha \in (0,1)$ makes the estimate track non-stationary rewards at the cost of residual variance.

### Applying the Bandit to CES Elasticity

The arms are discrete elasticity values. Each arm corresponds to a fixed CES elasticity value. The reward is the CES utility from allocating with that elasticity at the current market state.

> __Elasticity Bandit Setup:__
>
> * **Arms:** a finite ordered grid $\mathcal{H} = \{\eta_1 < \eta_2 < \cdots < \eta_K\}$ on $[\eta_{\min}, \eta_{\max}]$ spanning from near-Leontief ($\eta \to 0$, complementary holdings) to near-linear ($\eta \to \infty$, perfect substitutes). The cardinality $K$ and spacing are design choices: too coarse and the bandit cannot resolve the optimal $\eta$ per regime; too fine and exploration takes longer to converge. The companion example fixes a specific grid.
> * **Reward:** CES utility $U_{\mathrm{CES}}(\mathbf{n}^{\star}(\eta))$ at the chosen $\eta$, where $\mathbf{n}^{\star}(\eta)$ is the optimal share-count vector returned by the CES allocator (S2 § CES) when run at elasticity $\eta$ against the current prices and preferences.
> * **Per-regime learning:** Partition the sentiment signal into three bins using a threshold $\theta$:
>   - **Bearish** ($\lambda_t > \theta$): separate bandit instance
>   - **Neutral** ($|\lambda_t| \leq \theta$): separate bandit instance
>   - **Bullish** ($\lambda_t < -\theta$): separate bandit instance
>
>   Each bin maintains its own arm means and pull counts. The bandit learns the best $\eta \in \mathcal{H}$ independently per regime.
> * **Comparison baseline:** The Session 2 heuristic $\eta(\lambda_t) = \eta_{\min} + (\eta_{\max} - \eta_{\min})/(1 + |\lambda_t|)$

The per-regime partition lets the bandit discover asymmetric structure. A bearish regime might call for aggressive diversification (small $\eta$) while a bullish regime tolerates concentration (large $\eta$). The heuristic treats all extreme sentiment identically because it depends on $|\lambda|$; the bandit can break this symmetry.

### Algorithm: Per-Regime Elasticity Bandit (Online)

__Initialize__: Given an elasticity grid $\mathcal{H} = \{\eta_1, \ldots, \eta_K\}$, a sentiment threshold $\theta > 0$, a regime set $\mathcal{R} = \{\mathrm{bear}, \mathrm{neutral}, \mathrm{bull}\}$, a sentiment series $\{\lambda_t\}_{t=1}^{T}$, a rebalance schedule $\{b_t\}_{t=1}^{T}$ with $b_t \in \{0, 1\}$ (using $b$ rather than $a$ so the symbol does not collide with the bandit arm $a_t$ defined above), and a CES utility oracle $U_{\mathrm{CES}}(\mathbf{n}^{\star}(\eta))$ that returns the realized utility when the allocator runs at elasticity $\eta$, initialize a bandit per regime: for each $\rho \in \mathcal{R}$ set a regime round counter $\tau_\rho \gets 0$, and for each $(\rho, \eta) \in \mathcal{R} \times \mathcal{H}$ set $\hat{\mu}_{\rho,\eta} \gets 0$ and $n_{\rho,\eta} \gets 0$.

For $t = 1, \ldots, T$ with $b_t = 1$ __do__:

1. Bin the current sentiment $\lambda_t$ into a regime:
    - If $\lambda_t > \theta$, set $\rho_t \gets \mathrm{bear}$.
    - Else if $\lambda_t < -\theta$, set $\rho_t \gets \mathrm{bull}$.
    - Otherwise, set $\rho_t \gets \mathrm{neutral}$.
2. Advance the regime round counter and compute the regime-local exploration rate: $\tau_{\rho_t} \gets \tau_{\rho_t} + 1$, $\varepsilon \gets \tau_{\rho_t}^{-1/3} \cdot (K \ln \tau_{\rho_t})^{1/3}$.
3. Draw $p \sim \mathrm{Uniform}[0,1]$ and select an arm from bandit $\rho_t$:
    - If $p \leq \varepsilon$, *explore*: sample $\eta_t$ uniformly from $\mathcal{H}$.
    - Otherwise, *exploit*: set $\eta_t \gets \arg\max_{\eta \in \mathcal{H}} \hat{\mu}_{\rho_t, \eta}$.
4. Solve the CES allocation at elasticity $\eta_t$ and observe the reward $r_t \gets U_{\mathrm{CES}}(\mathbf{n}^{\star}(\eta_t))$.
5. Update the pulled arm for the active regime:
    $$n_{\rho_t, \eta_t} \gets n_{\rho_t, \eta_t} + 1, \qquad \hat{\mu}_{\rho_t, \eta_t} \gets \hat{\mu}_{\rho_t, \eta_t} + \frac{1}{n_{\rho_t, \eta_t}}\left(r_t - \hat{\mu}_{\rho_t, \eta_t}\right)$$

__Output__: Return the learned elasticity map $\rho \mapsto \arg\max_{\eta \in \mathcal{H}} \hat{\mu}_{\rho, \eta}$ for $\rho \in \mathcal{R}$, together with the per-regime arm means $\{\hat{\mu}_{\rho,\eta}\}$, pull counts $\{n_{\rho,\eta}\}$, and the chosen-arm history $\{(\rho_t, \eta_t, r_t)\}$.

The same epsilon-greedy framework extends naturally to asset selection: with $K$ assets, the $2^K - 1$ non-empty subsets form the arms, and the Cobb-Douglas utility over the included assets is the reward. This is a combinatorially harder problem, but the exploration-exploitation logic is identical. The combinatorial extension is treated separately in the optional materials.

> __Example__
>
> [▶ Let's run the elasticity bandit per regime and compare to the heuristic](eCornell-AI-Finance-S3-Example-Core-BanditEtaLearning-May-2026.ipynb). We fix the discrete $\eta$ grid, train separate bandits per sentiment bin on the Monte Carlo ensemble, visualize convergence and regret, and compare the learned elasticity map to the Session 2 formula.

___

## Validation and Compliance

Before the engine goes to production in Session 4, it must pass a formal validation. The validation report applies five pass/fail gates to the Monte Carlo backtest results:

| Criterion | Threshold | Rationale |
|-----------|-----------|-----------|
| **Median [Sharpe](../session-1/eCornell-AI-Finance-S1-Lecture-StressTestingMinVariancePortfolios-May-2026.ipynb#the-capital-allocation-line-and-tangent-portfolio)** $\geq$ **0.3** | Risk-adjusted growth must exceed a low bar | Below 0.3, the strategy is not compensating for the risk it takes |
| **Median [Max Drawdown](../session-1/eCornell-AI-Finance-S1-Lecture-StressTestingMinVariancePortfolios-May-2026.ipynb#portfolio-npv-and-tail-risk-metrics)** $\leq$ **25%** | Worst-case capital loss must be bounded | Larger drawdowns trigger investor redemptions and regulatory scrutiny |
| **Failure Rate** $\leq$ **10%** | At most 10% of paths end below initial capital | Systematic losses indicate a flawed strategy |
| **Wealth versus Frozen Baseline** $\geq$ **1.0** | Median terminal wealth must beat the frozen-parameter engine | An online learner that does not beat its frozen ancestor is not earning its complexity |
| **Median NPV** $\geq$ **0** | Median path must beat the risk-free baseline in present-value terms | A strategy whose median NPV is negative is dominated by a T-bill |

A strategy that passes all five gates across model-generated paths has demonstrated robustness to regime shifts, fat tails, and volatility clustering. It is not guaranteed to work in production, but it has cleared the minimum bar for deployment.

The validation report also produces a set of **compliance parameters** that become the pre-approved operating limits for Session 4's production system:

* **Concentration cap:** maximum portfolio weight in any single asset (e.g., 40%)
* **Drawdown gate:** maximum drawdown before the engine de-risks to cash (e.g., 15%)
* **Turnover limit:** maximum daily trade volume as a fraction of portfolio value (e.g., 50%)
* **Position size limit:** maximum dollar exposure per individual trade (e.g., $5,000)

In Session 4, trades within these limits auto-execute; trades that exceed them are queued for human review. The compliance configuration is the bridge between _validated in simulation_ and _approved for production._

> __Example__
>
> [▶ Let's build the validation report and export compliance config](eCornell-AI-Finance-S3-Example-Core-ValidationReport-May-2026.ipynb). We evaluate the EWLS + bandit engine against the five gates, produce pass/fail verdicts, and export the compliance parameters for Session 4.

___

## Summary

In this session, we replaced frozen parameters with online estimation and a hand-tuned heuristic with learned elasticity values, then validated the results against formal deployment gates.

> __Key Takeaways:__
>
> * __EWLS tracks parameter drift online:__ Exponentially weighted least squares re-estimates the SIM intercept, slope, and residual standard deviation daily with exponential decay, so preference weights adapt to regime changes without batch re-calibration. The half-life and prior weight control the speed-stability trade-off.
> * __Bandits learn elasticity per regime:__ The epsilon-greedy bandit discovers the best CES elasticity from a discrete grid, with separate instances for bearish, neutral, and bullish sentiment bins. This replaces the hand-tuned heuristic with data-driven elasticity selection that can capture asymmetric regime structure.
> * __Validation gates enforce deployment readiness:__ Five pass/fail criteria (Sharpe, drawdown, failure rate, wealth versus the frozen baseline, and median NPV) produce a binary go/no-go decision for production deployment. The compliance configuration exports the operating limits that Session 4's two-tier execution system enforces in real time.

**Next Session:** In [Session 4](../session-4/eCornell-AI-Finance-S4-Lecture-ProductionOps-May-2026.ipynb), the validated engine moves to production with three operational layers:

* __Cron execution:__ A scheduled agent runs the engine during market hours, reading the same compliance limits defined in this session's validation report and submitting orders against the live broker.
* __Two-tier policy:__ Trades that fall inside the pre-approved compliance envelope (concentration, drawdown gate, turnover, position size) auto-execute; trades that exceed any limit queue for human review before they can be released.
* __Post-market review:__ The evening session walks the day's audit trail, P&L attribution, and escalation log so every order, fill, and queued proposal is accounted for before the next open.

Friday's compliance config from Example 3 is the artifact that makes all three layers possible.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The models and strategies described are pedagogical tools using synthetic data. Real-world deployment requires additional regulatory, legal, and operational considerations.

___